# 🛸 DroneNet-FPN-Attention: Kaggle Dual-GPU DDP Training Pipeline
### ISLab Pusan National University — AI Engineer / Researcher Assignment
**Author:** Ghiffari Ahmadijaya (`ghiffariahmadijaya@gmail.com`)

**Key Capabilities:**
- 🚀 **Multi-GPU DistributedDataParallel (DDP)** on Dual Tesla T4 GPUs
- 🔍 **100% From-Scratch Vanilla Architecture** (Zero Pretrained Weights)
- 🎯 **High-Resolution FPN (P2/P3/P4) + RFB + Coordinate Attention**
- 📥 **Auto-Download Google Drive Dataset** (No Local Upload Needed)
- ⚡ **High-Speed Pre-Caching** (~3s/epoch training speed)
- 📊 **Full Evaluation & IEEE Paper Compilation in Cloud**

In [ ]:
# 1. Setup Environment & Check GPUs
import os, sys
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
os.environ["PYTHONWARNINGS"] = "ignore"
import torch

!nvidia-smi
print(f"Python Version : {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")
num_gpus = torch.cuda.device_count()
print(f"Device Count   : {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# 2. Install Dependencies Quietly
!pip install -q --no-cache-dir gdown tabulate pyyaml wandb tensorboard reportlab pypdf onnx onnxruntime onnxscript

In [ ]:
# 3. Clone Repository from GitHub into Working Directory
import os, shutil
os.chdir("/kaggle/working")
if not os.path.exists("src"):
    print("Cloning codebase from GitHub: https://github.com/itanium-g/islab-pusan-ai-assignment.git...")
    !git clone https://github.com/itanium-g/islab-pusan-ai-assignment.git /tmp/repo
    !cp -r /tmp/repo/* .
    !cp -r /tmp/repo/.* . 2>/dev/null || true
    !rm -rf /tmp/repo
print("Codebase initialized successfully.")
print("Directory contents:", os.listdir("/kaggle/working"))

In [ ]:
# 4. Download and Extract Dataset to /tmp to Keep /kaggle/working Clean
import os, zipfile, shutil, glob, gdown
GDRIVE_ID = "19L9yUP62xMESJMw6srf5HGcL8s5b0gv8"
TMP_DATA_DIR = "/tmp/curated_datasets/obj_det_base"
os.makedirs(TMP_DATA_DIR, exist_ok=True)

existing_txts = glob.glob(f"{TMP_DATA_DIR}/*.txt")
if len(existing_txts) < 100:
    print("Downloading dataset from Google Drive...")
    gdown.download(id=GDRIVE_ID, output="/tmp/dataset.zip", quiet=False)
    print("Extracting dataset.zip...")
    with zipfile.ZipFile("/tmp/dataset.zip", "r") as zf:
        zf.extractall("/tmp/extracted")
    if os.path.exists("/tmp/dataset.zip"):
        os.remove("/tmp/dataset.zip")
    # Move extracted files to TMP_DATA_DIR
    for root, dirs, files in os.walk("/tmp/extracted"):
        if any(f.endswith(".txt") for f in files):
            for f in files:
                src_p = os.path.join(root, f)
                dst_p = os.path.join(TMP_DATA_DIR, f)
                if not os.path.exists(dst_p):
                    shutil.move(src_p, dst_p)
            break
    if os.path.exists("/tmp/extracted"):
        shutil.rmtree("/tmp/extracted")

total_files = len(glob.glob(f"{TMP_DATA_DIR}/*.txt"))
print(f"Verified dataset: {total_files} annotation files in {TMP_DATA_DIR}")

# Create symlink in working directory for seamless path access
os.makedirs("curated_datasets", exist_ok=True)
if not os.path.exists("curated_datasets/obj_det_base"):
    os.symlink(TMP_DATA_DIR, "curated_datasets/obj_det_base")
print("Symlinked curated_datasets/obj_det_base ->", TMP_DATA_DIR)

In [ ]:
# 5. Preprocess & Cache Dataset (Reduces Epoch Time from 3 min to 3s!)
!python scripts/split_dataset.py --dataset-dir curated_datasets/obj_det_base --output-dir data/splits
!python scripts/preprocess_dataset.py --src-dir curated_datasets/obj_det_base --dest-dir data/cached_640 --img-size 640 --workers 8

In [ ]:
# 6. Train Model 3 (DroneNet-FPN-Attention - BEST MODEL) with Multi-GPU DDP Engine
import torch
num_gpus = torch.cuda.device_count()
if num_gpus > 1:
    print(f"Launching Multi-GPU DDP Training across {num_gpus} GPUs...")
    !torchrun --nproc_per_node={num_gpus} train.py --config configs/model3_fpn_attn.yaml --epochs 40 --batch-size 16 --ddp
elif num_gpus == 1:
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Launching Training on Single GPU ({gpu_name}) with AMP...")
    !python train.py --config configs/model3_fpn_attn.yaml --epochs 40 --batch-size 16
else:
    print("Training on CPU...")
    !python train.py --config configs/model3_fpn_attn.yaml --epochs 5 --batch-size 8

In [ ]:
# 7. Train Baseline Model (Model 1) & Multi-Scale FPN (Model 2) for Ablation Benchmark
!python train.py --config configs/model1_baseline.yaml --epochs 30 --batch-size 16
!python train.py --config configs/model2_fpn.yaml --epochs 35 --batch-size 16

In [ ]:
# 8. Comprehensive Benchmark Evaluation on Test Set across All 3 Models
print("=== EVALUATING MODEL 1 (BASELINE) ===")
!python evaluate.py --config configs/model1_baseline.yaml --weights runs/train/model1_vanilla_baseline/checkpoints/best_model.pth --split test

print("\n=== EVALUATING MODEL 2 (MULTI-SCALE FPN) ===")
!python evaluate.py --config configs/model2_fpn.yaml --weights runs/train/model2_fpn_multiscale/checkpoints/best_model.pth --split test

print("\n=== EVALUATING MODEL 3 (FPN + ATTENTION - BEST MODEL) ===")
!python evaluate.py --config configs/model3_fpn_attn.yaml --weights runs/train/model3_fpn_attention_best/checkpoints/best_model.pth --split test

In [ ]:
# 9. Export Lightweight Stripped Weights (< 15 MB), TorchScript & ONNX Models
os.makedirs("/kaggle/working/weights", exist_ok=True)
!python scripts/export_weights.py --config configs/model3_fpn_attn.yaml --checkpoint runs/train/model3_fpn_attention_best/checkpoints/best_model.pth --output-dir /kaggle/working/weights

In [ ]:
# 10. Generate Paper Figures and Compile IEEE PDF in Cloud
!python scripts/generate_paper_figures.py
!python scripts/compile_paper.py
!cp paper/paper.pdf /kaggle/working/Drone_Detection_Paper.pdf
print("Paper PDF saved to: /kaggle/working/Drone_Detection_Paper.pdf")

In [ ]:
# 11. Visual Inference & Sample Renders
import glob, os, shutil
from PIL import Image
import matplotlib.pyplot as plt

os.makedirs("/kaggle/working/inference_renders", exist_ok=True)
!python infer.py --config configs/model3_fpn_attn.yaml --weights runs/train/model3_fpn_attention_best/checkpoints/best_model.pth --source data/cached_640/images --output-dir /kaggle/working/inference_renders

sample_renders = glob.glob("/kaggle/working/inference_renders/*.jpg")[:3]
for p in sample_renders:
    plt.figure(figsize=(10, 6))
    plt.imshow(Image.open(p))
    plt.title(f"Detection Output: {os.path.basename(p)}")
    plt.axis("off")
    plt.show()

In [ ]:
# 12. Final Cleanup of Heavy Temporary Cache to Keep Output Package Lightweight (< 20 MB)
import os, shutil
if os.path.islink("curated_datasets/obj_det_base"):
    os.unlink("curated_datasets/obj_det_base")
if os.path.exists("data/cached_640"):
    shutil.rmtree("data/cached_640")
print("Kaggle working directory output files:")
for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        p = os.path.join(root, f)
        sz = os.path.getsize(p) / (1024 * 1024)
        if not any(x in root for x in ["__pycache__", ".git"]):
            print(f"  {p} ({sz:.2f} MB)")